In [1]:
from benchmark_pipeline.path import DATA
from benchmark_pipeline.config import (
    FEATURES,
    CITY_BBOXES,
    CITY_ECRS,
)
from benchmark_pipeline.generator.features import *

from benchmark_pipeline.loader.osm_loaders import load_tables
from benchmark_pipeline.loader.data_cleaning.name_matching import name_matching_cleaning
from benchmark_pipeline.generator.composite.generator_qcomposite import make_question_composite
from benchmark_pipeline.generator.composite.format_samples import format_composit_samples
from benchmark_pipeline.generator.register import register, BenchmarkConfig
from benchmark_pipeline.generator.multihop.generator_multihop import make_multihop_dataset

from benchmark_pipeline.generator.targets.mask_anchors import add_anchors_mask
from benchmark_pipeline.generator.targets.ellypses import add_ellypse
from benchmark_pipeline.generator.targets.anchors import add_anchors
from benchmark_pipeline.dataviz_tools.question_viz import montrer_questions

In [3]:
import pandas as pd
dataset = pd.read_parquet("/Users/sese/Documents/code/benchmark_pipeline/data/Paris/multihop/df_multihop.parquet")

In [ ]:
"Cafe to the south of restaurant and to the south of Le Bosphore and at less than 500 meters from Maison Blanche. "
"Restaurant at less than 100 meters from cafe and at the intersection of Rue Guyton de Morveau and Rue de l'Espérance and near Hôtel Ibis Italie Tolbiac towards La terrasse d'Italie. "
"Cafe to the west of bar and near Boba Tea Coffee towards Café Canon and near Mercure Gobelins towards Buja. "
"Bar at less than 1000 meters from hotel and at the west of Quartier de la Gare and near Porte d'Ivry. 
Hotel to the west of restaurant and at less than 500 meters from Pho Tai and at less than 100 meters from Pho Tai. 
Restaurant inside Petit-Ivry and near Viet Gourmet and at the border of Paris 13e Arrondissement.

In [2]:
import time

OVERWRITE = False   # True regenerates the cities already done, False skips them

for city_name, bbox in CITY_BBOXES.items():
    city_start = time.perf_counter()
    print(f"\n=== [{list(CITY_BBOXES).index(city_name) + 1}/{len(CITY_BBOXES)}] {city_name} ===")
    # A city is done once the last file of the loop, written by make_multihop_dataset, exists:
    # a city stopped halfway (folder without multihop/) is generated again
    if not OVERWRITE and (DATA / city_name / "multihop" / "df_multihop.parquet").exists():
        print("    déjà générée, ignorée (OVERWRITE = False)")
        continue
    conf = BenchmarkConfig(name=city_name, bbox=bbox, ecrs=CITY_ECRS[city_name], rows_target=[])

    print("1/5 chargement OSM...")
    step_start = time.perf_counter()
    df_osm, df_street, df_area = load_tables(conf.bbox, conf.ecrs, city_name=conf.name)
    print(f"    {len(df_osm)} POIs, {len(df_street)} rues, {len(df_area)} zones ({time.perf_counter() - step_start:.0f} s)")

    print("2/5 dédoublonnage des noms...")
    df_all, (df_osm_clean, df_street_clean, df_area_clean) = name_matching_cleaning(df_osm, df_street, df_area)
    print(f"    gardés : {len(df_osm_clean)} POIs, {len(df_street_clean)} rues, {len(df_area_clean)} zones")

    print("3/5 génération des questions...")
    step_start = time.perf_counter()
    df_question = make_question_composite(
        df_osm_clean, 
        df_area_clean, 
        df_street_clean,
        nb_q_by_temp=conf.nb_q_by_temp, 
        nb_q_by_feat=conf.nb_q_by_feat,
        ratio_features=conf.ratio_features, 
        part_avec_features=conf.part_with_features,
        feature_cols=FEATURES, 
        seed=conf.seed, 
        is_paraphrase=conf.is_paraphrase
    )
    print(f"    {len(df_question)} questions ({time.perf_counter() - step_start:.0f} s)")

    print("4/5 enregistrement du benchmark...")
    step_start = time.perf_counter()
    register(
        DATA / conf.name, 
        df_question, 
        df_osm_clean, 
        df_street_clean, 
        df_area_clean, 
        df_all,
        conf, 
        format_composit_samples, 
        overwritte=True
    )
    print(f"    écrit dans {DATA / conf.name} ({time.perf_counter() - step_start:.0f} s)")

    print("5/5 questions multihop...")
    step_start = time.perf_counter()
    make_multihop_dataset(
        conf.nb_seed_questions, 
        conf.name, 
        conf.multihop_template,
        df_question, 
        df_osm_clean, 
        df_street_clean, 
        df_area_clean, 
        seed=conf.seed,
        nb_q_by_feat=conf.nb_q_by_feat, 
        ratio_features=conf.ratio_features,
        part_with_features=conf.part_with_features
    )
    print(f"    fait ({time.perf_counter() - step_start:.0f} s)")
    print(f"=== {city_name} terminé en {(time.perf_counter() - city_start) / 60:.1f} min ===")



=== [1/9] Paris ===
1/5 chargement OSM...


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


75 morceaux de rue recollés
    18255 POIs, 8049 rues, 1217 zones (5 s)
2/5 dédoublonnage des noms...
lignes supprimées par source : {'street': 2, 'poi': 2}
    gardés : 16012 POIs, 7028 rues, 1211 zones
3/5 génération des questions...
    39765 questions (34 s)
4/5 enregistrement du benchmark...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/Paris (28 s)
5/5 questions multihop...
    fait (453 s)
=== Paris terminé en 8.8 min ===

=== [2/9] New York ===
1/5 chargement OSM...


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


2475 morceaux de rue recollés
    15932 POIs, 26034 rues, 2565 zones (15 s)
2/5 dédoublonnage des noms...
lignes supprimées par source : {'poi': 2, 'street': 1}
    gardés : 13716 POIs, 16124 rues, 2565 zones
3/5 génération des questions...
    25297 questions (35 s)
4/5 enregistrement du benchmark...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/New York (15 s)
5/5 questions multihop...
    fait (258 s)
=== New York terminé en 5.5 min ===

=== [3/9] Tokyo ===
1/5 chargement OSM...


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


... (9 outputs left out)

/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


1865 morceaux de rue recollés
    1587 POIs, 4389 rues, 337 zones (3 s)
2/5 dédoublonnage des noms...
lignes supprimées par source : {'poi': 1}
    gardés : 1451 POIs, 1971 rues, 324 zones
3/5 génération des questions...
    18145 questions (10 s)
4/5 enregistrement du benchmark...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/Medellin (8 s)
5/5 questions multihop...
    fait (145 s)
=== Medellin terminé en 2.8 min ===

=== [9/9] Marrakesh ===
1/5 chargement OSM...


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


29 morceaux de rue recollés
    930 POIs, 645 rues, 28 zones (1 s)
2/5 dédoublonnage des noms...
lignes supprimées par source : {'street': 1, 'poi': 1}
    gardés : 887 POIs, 622 rues, 28 zones
3/5 génération des questions...
    13589 questions (6 s)
4/5 enregistrement du benchmark...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/Marrakesh (6 s)
5/5 questions multihop...
    fait (191 s)
=== Marrakesh terminé en 3.4 min ===


In [3]:
"""    #data= make_paraphrase(df_question_test, nb_generated=10)

    is_gseek_training = False
    if is_gseek_training:
        #sentence_model = SentenceEncoder()
        #df_question = add_anchors_mask(df_question, sentence_model)
        df_question = add_ellypse(df_question)
        df_question = add_anchors(df_question)
    #df_question = df_question.rename(columns={"pois": "results_poi_id"})
    # """

'    #data= make_paraphrase(df_question_test, nb_generated=10)\n\n    is_gseek_training = False\n    if is_gseek_training:\n        #sentence_model = SentenceEncoder()\n        #df_question = add_anchors_mask(df_question, sentence_model)\n        df_question = add_ellypse(df_question)\n        df_question = add_anchors(df_question)\n    #df_question = df_question.rename(columns={"pois": "results_poi_id"})\n    # '

In [4]:
#montrer_questions(df_question, df_osm, par_type=True)